# NMF on ModulAir 00677

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00677
data = pd.read_csv('MOD-00677.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-30T02:01:54Z,576295658,2025-12-29T21:01:54Z,MOD-00677,38.3,1.9,1.406,0.240,0.072,0.025,0.011,...,33.070,30.471,14304.0,14305.0,14306.0,14417.0,14416.0,14420.0,14423.0,5.40
2025-12-30T02:00:54Z,576295660,2025-12-29T21:00:54Z,MOD-00677,37.8,2.0,1.506,0.250,0.084,0.007,0.015,...,33.099,31.187,14304.0,14305.0,14306.0,14417.0,14416.0,14420.0,14423.0,9.71
2025-12-30T01:59:54Z,576295659,2025-12-29T20:59:54Z,MOD-00677,37.3,2.0,1.327,0.223,0.097,0.007,0.015,...,32.646,31.564,14304.0,14305.0,14306.0,14417.0,14416.0,14420.0,14423.0,10.61
2025-12-30T01:58:54Z,576295657,2025-12-29T20:58:54Z,MOD-00677,36.8,2.0,1.364,0.234,0.080,0.010,0.017,...,32.904,31.616,14304.0,14305.0,14306.0,14417.0,14416.0,14420.0,14423.0,10.13
2025-12-30T01:57:54Z,576293842,2025-12-29T20:57:54Z,MOD-00677,36.3,2.0,1.279,0.255,0.103,0.027,0.014,...,31.740,31.343,14304.0,14305.0,14306.0,14417.0,14416.0,14420.0,14423.0,14.81


In [3]:
start = data.index.min()
end = data.index.max()

print(start, end)

2025-04-01T00:00:59Z 2025-12-30T02:01:54Z


In [7]:
len(data)

392270

In [8]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-29 21:01:54,683.923,1.886,33.070,30.471,1.406,0.240,0.072,0.025,0.011,0.018
1,2025-12-29 21:00:54,686.849,1.822,33.099,31.187,1.506,0.250,0.084,0.007,0.015,0.011
2,2025-12-29 20:59:54,680.225,1.951,32.646,31.564,1.327,0.223,0.097,0.007,0.015,0.018
3,2025-12-29 20:58:54,680.170,1.953,32.904,31.616,1.364,0.234,0.080,0.010,0.017,0.011
4,2025-12-29 20:57:54,677.241,1.823,31.740,31.343,1.279,0.255,0.103,0.027,0.014,0.010


In [9]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-29 21:01:54,683.923,1.886,33.070,30.471,1.406,0.240,0.072,0.025,0.011,0.018
1,2025-12-29 21:00:54,686.849,1.822,33.099,31.187,1.506,0.250,0.084,0.007,0.015,0.011
2,2025-12-29 20:59:54,680.225,1.951,32.646,31.564,1.327,0.223,0.097,0.007,0.015,0.018
3,2025-12-29 20:58:54,680.170,1.953,32.904,31.616,1.364,0.234,0.080,0.010,0.017,0.011
4,2025-12-29 20:57:54,677.241,1.823,31.740,31.343,1.279,0.255,0.103,0.027,0.014,0.010


In [10]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-29 21:01:54,683.923,1.886,33.070,30.471,1.406,0.240,0.072,0.025,0.011,0.018
1,2025-12-29 21:00:54,686.849,1.822,33.099,31.187,1.506,0.250,0.084,0.007,0.015,0.011
2,2025-12-29 20:59:54,680.225,1.951,32.646,31.564,1.327,0.223,0.097,0.007,0.015,0.018
3,2025-12-29 20:58:54,680.170,1.953,32.904,31.616,1.364,0.234,0.080,0.010,0.017,0.011
4,2025-12-29 20:57:54,677.241,1.823,31.740,31.343,1.279,0.255,0.103,0.027,0.014,0.010


In [11]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-03-31 20:00:00,828.62,32.20,37.44,1.86,16.19,2.00,1.06,0.46,0.68,0.48
1,2025-03-31 21:00:00,972.90,38.77,33.22,2.55,25.68,4.03,1.59,0.57,0.81,0.60
2,2025-03-31 22:00:00,1123.46,42.09,34.06,2.94,28.25,8.16,2.45,0.71,0.85,0.59
3,2025-03-31 23:00:00,852.74,24.68,50.46,2.79,16.52,4.15,1.30,0.34,0.36,0.22
4,2025-04-01 00:00:00,729.83,27.89,56.35,2.73,12.95,2.74,0.79,0.20,0.22,0.14
...,...,...,...,...,...,...,...,...,...,...,...
6537,2025-12-29 17:00:00,665.76,30.38,31.64,2.20,1.37,0.27,0.11,0.02,0.02,0.01
6538,2025-12-29 18:00:00,678.24,31.42,30.51,2.01,1.55,0.29,0.11,0.02,0.02,0.01
6539,2025-12-29 19:00:00,668.57,31.07,30.25,1.93,1.33,0.27,0.10,0.02,0.02,0.01
6540,2025-12-29 20:00:00,676.10,32.08,31.10,1.89,1.24,0.24,0.09,0.02,0.02,0.01


In [12]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,828.62,32.20,37.44,1.86,16.19,2.00,1.06,0.46,0.68,0.48
2025-03-31 21:00:00,972.90,38.77,33.22,2.55,25.68,4.03,1.59,0.57,0.81,0.60
2025-03-31 22:00:00,1123.46,42.09,34.06,2.94,28.25,8.16,2.45,0.71,0.85,0.59
2025-03-31 23:00:00,852.74,24.68,50.46,2.79,16.52,4.15,1.30,0.34,0.36,0.22
2025-04-01 00:00:00,729.83,27.89,56.35,2.73,12.95,2.74,0.79,0.20,0.22,0.14


In [13]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [14]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [15]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.287833,0.622463,0.413291,0.032938,0.219139,0.073937,0.080121,0.125683,0.223684,0.294479
2025-03-31 21:00:00,0.337951,0.749468,0.366707,0.045157,0.347591,0.148983,0.120181,0.155738,0.266447,0.368098
2025-03-31 22:00:00,0.390250,0.813648,0.375980,0.052063,0.382377,0.301664,0.185185,0.193989,0.279605,0.361963
2025-03-31 23:00:00,0.296212,0.477093,0.557015,0.049407,0.223606,0.153420,0.098262,0.092896,0.118421,0.134969
2025-04-01 00:00:00,0.253517,0.539146,0.622033,0.048344,0.175284,0.101294,0.059713,0.054645,0.072368,0.085890


In [16]:
data.to_csv(r'MOD-00677_timeseries_hourly_scaled.csv')

## Implementing NMF

In [17]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [18]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.302813,0.616548,0.419328,0.052095,0.162459,0.166506,0.145365,0.153407,0.190867,0.206935
2025-03-31 21:00:00,0.351452,0.743729,0.375389,0.060749,0.284236,0.250595,0.198307,0.199137,0.236519,0.252413
2025-03-31 22:00:00,0.390768,0.812299,0.380421,0.067652,0.366574,0.310094,0.238583,0.235762,0.275943,0.292812
2025-03-31 23:00:00,0.319403,0.472259,0.549852,0.049199,0.223024,0.139622,0.100779,0.096426,0.120326,0.135476
2025-04-01 00:00:00,0.327084,0.523454,0.600264,0.048966,0.163959,0.084597,0.058617,0.055328,0.074563,0.089561
...,...,...,...,...,...,...,...,...,...,...
2025-12-29 17:00:00,0.231228,0.587638,0.348243,0.035261,0.023538,0.002080,0.000828,0.001813,0.008466,0.016765
2025-12-29 18:00:00,0.232962,0.608148,0.336471,0.035638,0.025944,0.003912,0.001477,0.002073,0.008029,0.016051
2025-12-29 19:00:00,0.229894,0.601288,0.333471,0.035201,0.023013,0.002731,0.001062,0.001920,0.008005,0.015985


In [19]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.048177,0.031943,0.106832,0.050400
1,0.056879,0.062532,0.134271,0.033093
2,0.061109,0.082187,0.157334,0.029905
3,0.036943,0.041698,0.062236,0.087035
4,0.043129,0.026992,0.034541,0.094346
...,...,...,...,...
6444,0.051192,0.001097,0.000055,0.036573
6445,0.052939,0.002119,0.000000,0.032766
6446,0.052368,0.001457,0.000040,0.032566
6447,0.054029,0.000887,0.000000,0.033155


In [20]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [21]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,3.450488,11.468072,3.435495,0.551223,0.000000,0.000000,0.000000,0.026167,0.068274,0.158104
Feature 2,1.231196,0.488274,0.152398,0.133882,4.272688,1.845988,0.697108,0.290017,0.039208,0.000000
Feature 3,0.223744,0.453610,0.109035,0.110149,0.000000,1.006633,1.152261,1.336395,1.681746,1.755120
Feature 4,1.455359,0.000000,4.708367,0.188395,0.515453,0.000000,0.000000,0.002237,0.132185,0.234436


In [22]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-03-31 20:00:00,0.048177,0.031943,0.106832,0.050400
2025-03-31 21:00:00,0.056879,0.062532,0.134271,0.033093
2025-03-31 22:00:00,0.061109,0.082187,0.157334,0.029905
2025-03-31 23:00:00,0.036943,0.041698,0.062236,0.087035
2025-04-01 00:00:00,0.043129,0.026992,0.034541,0.094346
...,...,...,...,...
2025-12-29 17:00:00,0.051192,0.001097,0.000055,0.036573
2025-12-29 18:00:00,0.052939,0.002119,0.000000,0.032766
2025-12-29 19:00:00,0.052368,0.001457,0.000040,0.032566


In [23]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,3.450488,11.468072,3.435495,0.551223,0.000000,0.000000,0.000000,0.026167,0.068274,0.158104
Factor 2,1.231196,0.488274,0.152398,0.133882,4.272688,1.845988,0.697108,0.290017,0.039208,0.000000
Factor 3,0.223744,0.453610,0.109035,0.110149,0.000000,1.006633,1.152261,1.336395,1.681746,1.755120
Factor 4,1.455359,0.000000,4.708367,0.188395,0.515453,0.000000,0.000000,0.002237,0.132185,0.234436


In [24]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.550414,0.104036,0.013048,0.315761,0.016742
no,0.590652,0.075993,0.043148,0.274570,0.015636
no2,0.966221,0.021792,0.013972,0.000000,-0.001984
o3,0.346019,0.008131,0.004015,0.645001,-0.003165
bin0,0.000000,0.770112,0.000000,0.238547,-0.008659
bin1,0.000000,0.816053,0.307109,0.000000,-0.123162
bin2,0.000000,0.501216,0.571752,0.000000,-0.072968
bin3,0.038961,0.228743,0.727432,0.004530,0.000334
bin4,0.077229,0.023493,0.695447,0.203370,0.000461
bin5,0.141565,0.000000,0.574510,0.285507,-0.001582


In [25]:
results.to_csv(r'4_factor_results.csv')
comp.to_csv(r'4_factor_comp.csv')
res.to_csv(r'4_factor_resid.csv')